In [1]:
import pandas as pd

store = pd.read_excel(
    "clean_store_data.xlsx"
)

sku = pd.read_excel(
    "Missing SKU.xlsx",
    sheet_name="Sheet1"
)

print(store.shape)
print(sku.shape)

(34833, 32)
(329561, 5)


In [2]:
sku = sku.rename(
    columns={
        "STORE ID":"Store Id",
        "CATEGORY":"Category"
    }
)

print(sku.columns)

Index(['Store Id', 'DATE OF AUDIT', 'Category', 'SKU NAME', 'BARCODE'], dtype='object')


In [3]:
sku = sku.drop_duplicates()

print(
    "SKU rows after removing duplicates:",
    len(sku)
)

SKU rows after removing duplicates: 329561


In [4]:
# Check if store id and category create duplicate

(
sku
.groupby(
[
"Store Id",
"Category"
]
)
.size()
.sort_values(
ascending=False
)
.head(10)
)

Store Id  Category
26615     Hair_洗护发    188
2133      Hair_洗护发    164
9         Hair_洗护发    117
36240     Hair_洗护发    117
34353     Hair_洗护发    117
1954      Hair_洗护发    117
11769     Hair_洗护发    117
11266     Hair_洗护发    117
18        Hair_洗护发    117
31830     Hair_洗护发    117
dtype: int64

In [5]:
# Group SKU file before merge

sku_grouped = (

sku

.groupby(
[
"Store Id",
"Category"
]
)

.agg({

"SKU NAME":

lambda x:
" | ".join(
x.dropna()
.astype(str)
.unique()
),

"BARCODE":

lambda x:
" | ".join(
x.dropna()
.astype(str)
.unique()
),

"DATE OF AUDIT":

lambda x:
" | ".join(
x.dropna()
.astype(str)
.unique()
)

})

.reset_index()

)

print("Grouped shape:")
print(sku_grouped.shape)

sku_grouped.head()

Grouped shape:
(25122, 5)


,Store Id,Category,SKU NAME,BARCODE,DATE OF AUDIT
0,2,Shaving_剃须刀,锋速3经典刀架及须泡组合装 | 吉列锋隐超值男士剃须套装 | 吉列锋速3刀片(6刀头),6900068803411 | 6900068806054 | 7702018026173,2017-07-19
1,2,Skin_护肤,玉兰油多效修护霜50g | 玉兰油多效修护防晒霜50g | 玉兰油®多效修护醒肤水 | 玉兰...,6903148043257 | 6903148043264 | 6903148085240 ...,2017-07-19
2,3,Baby_婴儿纸尿裤,帮宝适超薄干爽系列大包装中号62片 2015 | 帮宝适超薄干爽系列大包装初生型86片 20...,6903148158289 | 6903148158364 | 6903148204870 ...,2017-07-07
3,3,Fem_妇女卫生护理,护舒宝瞬洁丝薄日用18片 | 护舒宝瞬洁贴身量多日用/夜用12片卫生巾 | 护舒宝瞬洁贴身日...,6903148031971 | 6903148032053 | 6903148032084 ...,2017-07-07
4,3,Hair_洗护发,飘柔家庭护理绿茶长效清爽去油洗发露400ml | 海飞丝去屑洗发露清爽去油型400毫升 | ...,6903148030479 | 6903148045046 | 6903148045053 ...,2017-07-07


In [6]:
merged_final = pd.merge(
store,
sku_grouped,
on=[
"Store Id",
"Category"
],
how="left"
)
print("Rows:", len(merged_final))

Rows: 34833


In [7]:
print(

merged_final[
[
"SKU NAME",
"BARCODE",
"DATE OF AUDIT"
]
]

.isnull()

.sum()

)

SKU NAME         7528
BARCODE          7528
DATE OF AUDIT    7528
dtype: int64


In [8]:
matched = (

merged_final[
"SKU NAME"
]

.notnull()

.sum()

)

total = len(merged_final)

print(
"Match %:",
round(
matched/total*100,
2
)
)

Match %: 78.39


In [11]:
print(merged_final.columns)
print("\n\n")
print(merged_final.head())

Index(['Store Type', 'Category', 'Store Name', 'Store Id', 'Division',
       'retailer_name', 'Region', 'Banner Region', 'Province', 'City',
       'STAR-5', 'STAR-3', 'PSTAR-5', 'PSTAR-3', 'Length%_2017-05-01 00:00:00',
       'Length%_2017-06-01 00:00:00', 'Length%_2017-07-01 00:00:00',
       'Facings%_2017-05-01 00:00:00', 'Facings%_2017-06-01 00:00:00',
       'Facings%_2017-07-01 00:00:00', 'DisplayS%_2017-05-01 00:00:00',
       'DisplayS%_2017-06-01 00:00:00', 'DisplayS%_2017-07-01 00:00:00',
       'PSKU%_2017-05-01 00:00:00', 'PSKU%_2017-06-01 00:00:00',
       'PSKU%_2017-07-01 00:00:00', 'Star_2017-05-01 00:00:00',
       'Star_2017-06-01 00:00:00', 'Star_2017-07-01 00:00:00',
       'Star_Star JAS Base', 'STAR5_Achievement', 'STAR3_Achievement',
       'SKU NAME', 'BARCODE', 'DATE OF AUDIT'],
      dtype='object')



  Store Type    Category                   Store Name  Store Id    Division  \
0        C&C  Baby_婴儿纸尿裤  11187 - 锦江麦德龙先购自运有限公司南京下关商场     11187  Division 1   

In [13]:
merged_final.to_excel(

"Final_Merged_Store_SKU.xlsx",

index=False

)

print(
"File saved successfully"
)

File saved successfully
